# Pipeline Dự báo Doanh thu & COGS — Stacking Ensemble (Leak-free)

Pipeline này được thiết kế để dự báo chuỗi thời gian dựa trên phương pháp supervised learning, sử dụng kiến trúc **Stacking** để kết hợp các mô hình khác nhau.

### Triết lý thiết kế "Leak-free":
1. **Tuning khách quan**: Tìm tham số (Optuna) bằng Cross-Validation 3-fold trên tập Train. Tuyệt đối không dùng tập Validation để tránh "tuning leakage".
2. **OOF Stacking**: Meta-model học cách blending dựa trên Out-of-Fold predictions của Level 0. 
3. **Fold-aware Scaling**: Chuẩn hóa dữ liệu (Scaler) được thực hiện độc lập cho từng fold.
4. **Diversity**: Sử dụng cả mô hình Tree (LGBM, XGBoost) và Linear (Ridge) ở Level 0 để tăng sự ổn định.

## 1. Chuẩn bị Dữ liệu

In [ ]:
# ── Cấu hình Logging (Ghi log ra console và file) ───────────────────────
import logging
import os
from pathlib import Path

LOG_DIR  = Path("logs")
LOG_DIR.mkdir(exist_ok=True)
LOG_FILE = LOG_DIR / "train_test.log"

logging.basicConfig(
    level    = logging.INFO,
    format   = "%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt  = "%Y-%m-%d %H:%M:%S",
    handlers = [
        logging.StreamHandler(),
        logging.FileHandler(LOG_FILE, mode="a", encoding="utf-8"),
    ],
)
log = logging.getLogger(__name__)
log.info(f"Logging initialised → {LOG_FILE.resolve()}")

In [ ]:
# ── Cấu hình MLflow Tracking ──────────────────────────────────────────
import mlflow
import mlflow.sklearn
import mlflow.lightgbm
import mlflow.xgboost

MLFLOW_EXPERIMENT = "revenue-cogs-stacking"
mlflow.set_tracking_uri("mlruns")          # Lưu local, đã ignore trong git
mlflow.set_experiment(MLFLOW_EXPERIMENT)

log.info(f"MLflow experiment: '{MLFLOW_EXPERIMENT}'  |  tracking uri: mlruns/")

In [ ]:
from feature_engineering import DataLoader, FeatureEngineer

In [ ]:
# Load và build feature matrix
data     = DataLoader()
features = FeatureEngineer(data)

df = features.build()
log.info(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} cols")
df.head()

In [ ]:
df.to_csv("train_features.csv", index=False)
log.info("Saved feature matrix to train_features.csv")

## 2. Chia Train / Validation (Time-series Split)

Sử dụng chiến lược chia theo mốc thời gian để mô phỏng thực tế. 
- **Train set**: Dữ liệu đến hết 2021.
- **Validation set**: Năm 2022 (dùng để đo lường độ chính xác cuối cùng).

In [ ]:
TRAIN_END = "2021-12-31"
VAL_START = "2022-01-01"
VAL_END   = "2022-12-31"
log.info(f"Time split — train: ≤{TRAIN_END}  |  val: {VAL_START} → {VAL_END}")

In [ ]:
df_train = df[df.Date <= TRAIN_END]
df_val   = df[(df.Date >= VAL_START) & (df.Date <= VAL_END)]
log.info(f"df_train: {len(df_train):,} rows  |  df_val: {len(df_val):,} rows")

In [ ]:
FEATURES = [
    # Thời gian
    "year","month","day","day_of_week","day_of_year",
    "week_of_year","quarter",
    "is_weekend","is_month_end","is_month_start",
    "is_year_end","is_year_start",
    "sin_month","cos_month","sin_dow","cos_dow",
    "days_to_tet",

    # Biến Lag (Dùng giá trị quá khứ để dự báo)
    "revenue_lag_1","revenue_lag_2","revenue_lag_6",
    "revenue_lag_7","revenue_lag_14","revenue_lag_30",
    "revenue_lag_90","revenue_lag_365",
    "cogs_lag_1","cogs_lag_7","cogs_lag_30","cogs_lag_365",

    # Rolling window (Trung bình động)
    "revenue_roll_mean_7","revenue_roll_std_7",
    "revenue_roll_mean_14","revenue_roll_std_14",
    "revenue_roll_mean_30","revenue_roll_std_30",
    "revenue_roll_mean_90","revenue_roll_std_90",
    "revenue_ewm_7","revenue_ewm_30",

    # Xu hướng & Biến động
    "revenue_diff_1","revenue_diff_7",
    "revenue_pct_change_7",
    "cogs_roll_mean_7","cogs_roll_mean_30",
]
TARGETS = ["Revenue", "COGS"]

X_train = df_train[FEATURES].copy()
y_train = df_train[TARGETS].copy()

X_val   = df_val[FEATURES].copy()
y_val   = df_val[TARGETS].copy()
log.info(f"X_train shape: {X_train.shape}")

# Bước 3 — Stacking Ensemble (Leak-free Pipeline)

Đây là bước trọng tâm, bao gồm Tuning và Huấn luyện các cấp của bộ Ensemble.

In [ ]:
import numpy as np
import pandas as pd
import warnings
import joblib
import json
warnings.filterwarnings("ignore")

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

log.info("Libraries imported successfully")

In [ ]:
def objective_lgb(trial, X, y, parent_run_id):
    """
    Hàm mục tiêu cho Optuna (LightGBM).
    Sử dụng TimeSeriesSplit nội bộ trên tập Train để tính CV MAE.
    """
    params = dict(
        n_estimators      = trial.suggest_int("n_estimators", 300, 1500),
        learning_rate     = trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        num_leaves        = trial.suggest_int("num_leaves", 20, 255),
        max_depth         = trial.suggest_int("max_depth", 3, 12),
        subsample         = trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree  = trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_alpha         = trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        reg_lambda        = trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
        random_state      = 42,
        n_jobs            = -1,
        verbose           = -1,
    )
    
    tscv = TimeSeriesSplit(n_splits=3) # Dùng 3-fold để cân bằng tốc độ/độ chính xác khi tuning
    scores = []
    for tr_idx, val_idx in tscv.split(X):
        m = LGBMRegressor(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        preds = m.predict(X.iloc[val_idx])
        scores.append(mean_absolute_error(y.iloc[val_idx], preds))
    
    cv_mae = np.mean(scores)
    
    # Log từng thử nghiệm vào MLflow dưới dạng nested run
    with mlflow.start_run(run_name=f"lgb_trial_{trial.number}", nested=True):
        mlflow.log_params(params)
        mlflow.log_metric("cv_mae", cv_mae)
    return cv_mae

def objective_xgb(trial, X, y, parent_run_id):
    """
    Hàm mục tiêu cho Optuna (XGBoost).
    """
    params = dict(
        n_estimators     = trial.suggest_int("n_estimators", 300, 1500),
        learning_rate    = trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        max_depth        = trial.suggest_int("max_depth", 3, 10),
        subsample        = trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.6, 1.0),
        random_state     = 42,
        n_jobs           = -1,
        tree_method      = "hist",
    )
    
    tscv = TimeSeriesSplit(n_splits=3)
    scores = []
    for tr_idx, val_idx in tscv.split(X):
        m = XGBRegressor(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        preds = m.predict(X.iloc[val_idx])
        scores.append(mean_absolute_error(y.iloc[val_idx], preds))
    
    cv_mae = np.mean(scores)
    with mlflow.start_run(run_name=f"xgb_trial_{trial.number}", nested=True):
        mlflow.log_params(params)
        mlflow.log_metric("cv_mae", cv_mae)
    return cv_mae

def run_tuning(objective_fn, X, y, name, n_trials=30, run_id=""):
    """Helper function để khởi tạo và chạy Optuna study."""
    log.info(f"[Optuna] Tuning {name} — {n_trials} trials")
    study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
    study.optimize(lambda t: objective_fn(t, X, y, run_id), n_trials=n_trials)
    log.info(f"[{name}] Best CV MAE: {study.best_value:,.02f}")
    return study.best_params, study.best_value

In [ ]:
with mlflow.start_run(run_name="stacking_cv_pipeline") as main_run:
    run_id = main_run.info.run_id
    mlflow.set_tags({"model_type": "stacking_diversity_oof", "cv": "3-fold-tscv"})

    # --- 1. OPTUNA TUNING (Chạy trên X_train để tìm Best Params) ---
    log.info("--- 1. Hyperparameter Tuning (Cross-Validation) ---")
    best_lgb_rev, _  = run_tuning(objective_lgb, X_train, y_train['Revenue'], "LGB-Rev", 30, run_id)
    best_xgb_rev, _  = run_tuning(objective_xgb, X_train, y_train['Revenue'], "XGB-Rev", 30, run_id)
    best_lgb_cogs, _ = run_tuning(objective_lgb, X_train, y_train['COGS'],    "LGB-Cogs", 30, run_id)
    best_xgb_cogs, _ = run_tuning(objective_xgb, X_train, y_train['COGS'],    "XGB-Cogs", 30, run_id)

    # --- 2. GENERATE OOF PREDICTIONS (Level 0) ---
    # Tạo dự đoán Out-of-Fold để meta-model học cách kết hợp các model nền.
    log.info("--- 2. Generating OOF Predictions (Level 0) ---")
    tscv = TimeSeriesSplit(n_splits=5)
    
    # DataFrames lưu kết quả OOF
    oof_rev = pd.DataFrame(index=X_train.index, columns=['lgb','xgb','ridge'])
    oof_cogs = pd.DataFrame(index=X_train.index, columns=['lgb','xgb','ridge'])

    for fold, (tr_i, val_i) in enumerate(tscv.split(X_train)):
        Xt, Xv = X_train.iloc[tr_i], X_train.iloc[val_i]
        yt_rev, yv_rev = y_train['Revenue'].iloc[tr_i], y_train['Revenue'].iloc[val_i]
        yt_cogs, yv_cogs = y_train['COGS'].iloc[tr_i], y_train['COGS'].iloc[val_i]
        
        # Fold-aware scaling (Fit scaler chỉ trên training fold hiện tại)
        scaler = StandardScaler(); Xt_sc = scaler.fit_transform(Xt); Xv_sc = scaler.transform(Xv)
        
        # Revenue Level 0 Models
        m_lgb = LGBMRegressor(**best_lgb_rev, random_state=42, verbose=-1).fit(Xt, yt_rev)
        m_xgb = XGBRegressor(**best_xgb_rev, random_state=42).fit(Xt, yt_rev)
        m_rid = Ridge(alpha=100.0).fit(Xt_sc, yt_rev)
        oof_rev.iloc[val_i, 0] = m_lgb.predict(Xv); oof_rev.iloc[val_i, 1] = m_xgb.predict(Xv); oof_rev.iloc[val_i, 2] = m_rid.predict(Xv_sc)
        
        # COGS Level 0 Models
        m_lgb = LGBMRegressor(**best_lgb_cogs, random_state=42, verbose=-1).fit(Xt, yt_cogs)
        m_xgb = XGBRegressor(**best_xgb_cogs, random_state=42).fit(Xt, yt_cogs)
        m_rid = Ridge(alpha=100.0).fit(Xt_sc, yt_cogs)
        oof_cogs.iloc[val_i, 0] = m_lgb.predict(Xv); oof_cogs.iloc[val_i, 1] = m_xgb.predict(Xv); oof_cogs.iloc[val_i, 2] = m_rid.predict(Xv_sc)
        
        log.info(f"  Fold {fold+1}/5 done")

    # --- 3. TRAIN META MODELS (Level 1) ---
    # Meta-model (Ridge) học cách điều chỉnh trọng số (weights) dựa trên OOF.
    log.info("--- 3. Training Meta Models on OOF dataset ---")
    mask_rev = oof_rev.notna().all(axis=1)
    meta_rev  = Ridge(alpha=1.0).fit(oof_rev[mask_rev], y_train['Revenue'][mask_rev])
    
    mask_cogs = oof_cogs.notna().all(axis=1)
    meta_cogs = Ridge(alpha=1.0).fit(oof_cogs[mask_cogs], y_train['COGS'][mask_cogs])
    
    log.info(f"Meta Weights Rev: {meta_rev.coef_.round(3).tolist()}")
    log.info(f"Meta Weights COGS: {meta_cogs.coef_.round(3).tolist()}")

    # --- 4. FINAL EVALUATION ON VAL SET (2022) ---
    # Đánh giá hiệu năng thực tế trên tập Validation hoàn toàn mới.
    log.info("--- 4. Final Evaluation on Validation Set (2022) ---")
    
    # Retrain level 0 trên toàn bộ X_train
    scaler_final = StandardScaler(); X_tr_sc = scaler_final.fit_transform(X_train); X_val_sc = scaler_final.transform(X_val)
    
    lgb_r = LGBMRegressor(**best_lgb_rev, verbose=-1).fit(X_train, y_train['Revenue'])
    xgb_r = XGBRegressor(**best_xgb_rev).fit(X_train, y_train['Revenue'])
    rid_r = Ridge(alpha=100.0).fit(X_tr_sc, y_train['Revenue'])
    
    val_meta_rev = np.column_stack([lgb_r.predict(X_val), xgb_r.predict(X_val), rid_r.predict(X_val_sc)])
    stack_rev_pred = meta_rev.predict(val_meta_rev)

    lgb_c = LGBMRegressor(**best_lgb_cogs, verbose=-1).fit(X_train, y_train['COGS'])
    xgb_c = XGBRegressor(**best_xgb_cogs).fit(X_train, y_train['COGS'])
    rid_c = Ridge(alpha=100.0).fit(X_tr_sc, y_train['COGS'])
    
    val_meta_cogs = np.column_stack([lgb_c.predict(X_val), xgb_c.predict(X_val), rid_c.predict(X_val_sc)])
    stack_cogs_pred = meta_cogs.predict(val_meta_cogs)

    # Tính Metrics
    def evaluate(y_true, y_pred, label=""):
        mae  = mean_absolute_error(y_true, y_pred)
        rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
        r2   = r2_score(y_true, y_pred)
        log.info(f"{label:<14s}  MAE={mae:>12,.0f}  RMSE={rmse:>12,.0f}  R²={r2:.4f}")
        return dict(mae=mae, rmse=rmse, r2=float(r2))

    log.info("=" * 70); log.info("EVALUATION — Validation set 2022"); log.info("=" * 70)
    log.info("--- Revenue ---")
    r_lgb = evaluate(y_val["Revenue"], val_meta_rev[:,0], "LGB")
    r_stack = evaluate(y_val["Revenue"], stack_rev_pred, "STACK")
    
    log.info("--- COGS ---")
    c_lgb = evaluate(y_val["COGS"], val_meta_cogs[:,0], "LGB")
    c_stack = evaluate(y_val["COGS"], stack_cogs_pred, "STACK")

    # Log Metrics quan trọng lên MLflow
    mlflow.log_metrics({"val_mae_rev": r_stack['mae'], "val_mae_cogs": c_stack['mae']})
    log.info(f"MLflow run {run_id} finished")

## 4. Huấn luyện Mô hình Cuối (Full Data 2013–2022)

Sau khi đã xác thực, ta gộp dữ liệu để có bộ mô hình mạnh nhất phục vụ dự báo tương lai.

In [ ]:
df_full = df[df.Date <= "2022-12-31"].dropna(subset=["revenue_lag_365"])
X_full = df_full[FEATURES]; y_full = df_full[TARGETS]
scaler_f = StandardScaler(); X_full_sc = scaler_f.fit_transform(X_full)

with mlflow.start_run(run_name="final_retrain"):
    lgb_rf = LGBMRegressor(**best_lgb_rev, verbose=-1).fit(X_full, y_full['Revenue'])
    xgb_rf = XGBRegressor(**best_xgb_rev).fit(X_full, y_full['Revenue'])
    rid_rf = Ridge(alpha=100.0).fit(X_full_sc, y_full['Revenue'])
    
    lgb_cf = LGBMRegressor(**best_lgb_cogs, verbose=-1).fit(X_full, y_full['COGS'])
    xgb_cf = XGBRegressor(**best_xgb_cogs).fit(X_full, y_full['COGS'])
    rid_cf = Ridge(alpha=100.0).fit(X_full_sc, y_full['COGS'])

    meta_X_f_rev = np.column_stack([lgb_rf.predict(X_full), xgb_rf.predict(X_full), rid_rf.predict(X_full_sc)])
    meta_final_rev = Ridge(alpha=1.0).fit(meta_X_f_rev, y_full['Revenue'])
    
    meta_X_f_cogs = np.column_stack([lgb_cf.predict(X_full), xgb_cf.predict(X_full), rid_cf.predict(X_full_sc)])
    meta_final_cogs = Ridge(alpha=1.0).fit(meta_X_f_cogs, y_full['COGS'])
    
    os.makedirs("models", exist_ok=True)
    joblib.dump(lgb_rf, "models/lgb_rev.pkl"); joblib.dump(xgb_rf, "models/xgb_rev.pkl"); joblib.dump(rid_rf, "models/rid_rev.pkl")
    joblib.dump(lgb_cf, "models/lgb_cogs.pkl"); joblib.dump(xgb_cf, "models/xgb_cogs.pkl"); joblib.dump(rid_cf, "models/rid_cogs.pkl")
    joblib.dump(meta_final_rev, "models/meta_rev.pkl"); joblib.dump(meta_final_cogs, "models/meta_cogs.pkl")
    joblib.dump(scaler_f, "models/scaler.pkl")
    log.info("Final models saved to models/")

## 5. Giải thích Mô hình bằng SHAP (Model Interpretation)

Sử dụng SHAP để hiểu các yếu tố nào ảnh hưởng mạnh nhất đến doanh thu và chi phí.

In [ ]:
import shap
import matplotlib.pyplot as plt

def log_shap_summary(model, X, target_name, model_name):
    """Tính SHAP values và log summary plot lên MLflow."""
    log.info(f"[SHAP] Explaining {model_name} for {target_name}...")
    
    # Dùng TreeExplainer tối ưu cho LightGBM
    explainer = shap.TreeExplainer(model)
    # Lấy 500 dòng cuối để đại diện cho xu hướng gần nhất
    X_sample = X.tail(500)
    shap_values = explainer.shap_values(X_sample)
    
    # Vẽ Summary Plot (Dot)
    plt.figure(figsize=(12, 10))
    shap.summary_plot(shap_values, X_sample, show=False)
    
    plot_path = f"shap_summary_{target_name}_{model_name}.png"
    plt.title(f"SHAP Summary: {target_name} ({model_name})")
    plt.tight_layout()
    plt.savefig(plot_path, dpi=150)
    mlflow.log_artifact(plot_path, artifact_path="plots")
    plt.show()
    log.info(f"[SHAP] {plot_path} saved and logged to MLflow")

with mlflow.start_run(run_name="model_interpretation"):
    # Giải thích mô hình Revenue
    log_shap_summary(lgb_rf, X_full, "Revenue", "LGBM")
    
    # Giải thích mô hình COGS
    log_shap_summary(lgb_cf, X_full, "COGS", "LGBM")